In [ ]:
# Set up dask cluster - DO NOT RUN!

# from dask.distributed import LocalCluster
# cluster = LocalCluster(n_workers=70, memory_limit='auto', processes=True) # Create cluster
# cluster.scheduler_address # Show cluster address


In [ ]:
# Connect to dask cluster

from dask.distributed import LocalCluster, Client

address = 'tcp://127.0.0.1:34217'
client = Client(address)
client

# Click on ports, below in terminal window
# Forward a port
# For port number, use number from 'Dashboard:' from client output


<Client: 'tcp://127.0.0.1:34217' processes=70 threads=140, memory=220.10 GiB>

2025-04-08 12:15:09,134 - distributed.client - ERROR - Failed to reconnect to scheduler after 30.00 seconds, closing client


In [ ]:
import os              
import urllib 

import dask.dataframe as dd

In [ ]:
url = 'https://arcticdata.io/metacat/d1/mn/v2/object/urn%3Auuid%3A27e4043d-75eb-4c4f-9427-0d442526c154'
msg = urllib.request.urlretrieve(url, "dg_soil_moisture.csv")

In [ ]:
# How to handle a data file that is too big to read in...
# Read in in chunks

import pandas as pd
pd.read_csv("dg_soil_moisture.csv", nrows = 10, encoding = 'ISO-8859-1')


,timestamp,year,doy,hour,minute,site,logger,port,sensor,sensorZ,m_soil,unit
0,2014-07-07 16:30:00,2014,188,16,30,MDF1,MDF1met,Port 3,5TM Moisture/Temp,-6,0.273,m³/m³ VWC
1,2014-07-07 16:30:00,2014,188,16,30,MDF1,MDF1met,Port 4,5TM Moisture/Temp,-11,0.345,m³/m³ VWC
2,2014-07-07 17:00:00,2014,188,17,0,LDF2,LDF2met,Port 3,5TM Moisture/Temp,-8,0.308,m³/m³ VWC
3,2014-07-07 17:00:00,2014,188,17,0,LDF2,LDF2met,Port 4,5TM Moisture/Temp,-13,0.325,m³/m³ VWC
4,2014-07-07 17:00:00,2014,188,17,0,MDF1,MDF1met,Port 3,5TM Moisture/Temp,-6,0.283,m³/m³ VWC
5,2014-07-07 17:00:00,2014,188,17,0,MDF1,MDF1met,Port 4,5TM Moisture/Temp,-11,0.346,m³/m³ VWC
6,2014-07-07 17:30:00,2014,188,17,30,LBR,LBRmet,Port 3,5TM Moisture/Temp,-23,0.302,m³/m³ VWC
7,2014-07-07 17:30:00,2014,188,17,30,LDF2,LDF2met,Port 3,5TM Moisture/Temp,-8,0.308,m³/m³ VWC
8,2014-07-07 17:30:00,2014,188,17,30,LDF2,LDF2met,Port 4,5TM Moisture/Temp,-13,0.326,m³/m³ VWC
9,2014-07-07 17:30:00,2014,188,17,30,MDF1,MDF1met,Port 3,5TM Moisture/Temp,-6,0.283,m³/m³ VWC


In [ ]:
df = dd.read_csv("dg_soil_moisture.csv", encoding = 'ISO-8859-1', blocksize = '20MB') # blocksize specifies size of blocks to read in

# npartitions tells you how many chunks dask partitioned it into
# dask is lazy, so when reading it doesn't actually read in/display the data until you do something that needs the data
df

# Here we run a function that requires the data, so now it will show the data
df.head()

# dask dataframe is an abstraction, but at the core it's pandas dataframes i.e. each chunk is a pandas dataframe
type(df)
type(df.head())

pandas.core.frame.DataFrame

In [7]:
averages = df.groupby('year').mean(numeric_only = True)
averages

,doy,hour,minute,sensorZ,m_soil
npartitions=1,,,,,
,float64,float64,float64,float64,float64
,...,...,...,...,...


In [ ]:
# once you run compute, you are pulling down data as pandas dataframe (or array, depending on the data type you started with) to your local environment
# i.e. no longer distributed to the cluster workers
# don't call compute until you are done processing everything
# can use 'persist' to get around this
averages.compute()

,doy,hour,minute,sensorZ,m_soil
year,,,,,
2014,276.852636,11.513992,15.001123,-11.998332,0.273744
2015,186.720383,11.500723,15.000000,-13.047899,0.263738
2016,183.497453,11.499777,14.999858,-15.000009,0.293595
2017,181.414843,11.499381,15.000144,-14.999981,0.266121
2018,201.824077,11.500796,15.000356,-15.435365,0.282395
2019,173.693311,11.498577,15.000000,-15.124516,0.222193
2020,138.806679,11.489825,14.999322,-15.200054,0.252467


# Dask arrays

In [11]:
import numpy as np
import dask.array as da

In [ ]:
data = np.arange(100_000).reshape(200, 500)
data

array([[    0,     1,     2, ...,   497,   498,   499],
       [  500,   501,   502, ...,   997,   998,   999],
       [ 1000,  1001,  1002, ...,  1497,  1498,  1499],
       ...,
       [98500, 98501, 98502, ..., 98997, 98998, 98999],
       [99000, 99001, 99002, ..., 99497, 99498, 99499],
       [99500, 99501, 99502, ..., 99997, 99998, 99999]], shape=(200, 500))

In [15]:
a = da.from_array(data, chunks = (100, 100))
a

dask.array<array, shape=(200, 500), dtype=int64, chunksize=(100, 100), chunktype=numpy.ndarray>

In [17]:
a.mean().compute()

np.float64(49999.5)

In [ ]:
# download red band
url = 'https://arcticdata.io/metacat/d1/mn/v2/object/urn%3Auuid%3Aac25a399-b174-41c1-b6d3-09974b161e5a'
msg = urllib.request.urlretrieve(url, "RU_ANS_TR2_FL005M_red.tif")


# download nir band
url = 'https://arcticdata.io/metacat/d1/mn/v2/object/urn%3Auuid%3A1762205e-c505-450d-90ed-d4f3e4c302a7'
msg = urllib.request.urlretrieve(url, "RU_ANS_TR2_FL005M_nir.tif")

In [21]:
import rioxarray as rioxr

red = rioxr.open_rasterio("RU_ANS_TR2_FL005M_red.tif", chunks = '15MB')
red

nir = rioxr.open_rasterio("RU_ANS_TR2_FL005M_nir.tif", chunks = '15MB')
nir

<xarray.DataArray (band: 1, y: 3499, x: 7443)> Size: 104MB
dask.array<open_rasterio-9f5d4f3721aeba15f4b738607af287b1<this-array>, shape=(1, 3499, 7443), dtype=float32, chunksize=(1, 1936, 1936), chunktype=numpy.ndarray>
Coordinates:
  * band         (band) int64 8B 1
  * x            (x) float64 60kB 6.22e+05 6.22e+05 ... 6.224e+05 6.224e+05
  * y            (y) float64 28kB 7.525e+06 7.525e+06 ... 7.525e+06 7.525e+06
    spatial_ref  int64 8B 0
Attributes:
    TIFFTAG_SOFTWARE:  pix4dmapper
    AREA_OR_POINT:     Area
    _FillValue:        -10000.0
    scale_factor:      1.0
    add_offset:        0.0

In [ ]:
# Gets rid of unnecessary single dimensions
red = red.squeeze() 
nir = nir.squeeze()

<xarray.DataArray (y: 3499, x: 7443)> Size: 104MB
dask.array<getitem, shape=(3499, 7443), dtype=float32, chunksize=(1936, 1936), chunktype=numpy.ndarray>
Coordinates:
    band         int64 8B 1
  * x            (x) float64 60kB 6.22e+05 6.22e+05 ... 6.224e+05 6.224e+05
  * y            (y) float64 28kB 7.525e+06 7.525e+06 ... 7.525e+06 7.525e+06
    spatial_ref  int64 8B 0
Attributes:
    TIFFTAG_SOFTWARE:  pix4dmapper
    AREA_OR_POINT:     Area
    _FillValue:        -10000.0
    scale_factor:      1.0
    add_offset:        0.0

In [ ]:
ndvi = (nir - red) / (nir + red)
ndvi # Lazy eval, just shows dask array with chunks again
ndvi_values = ndvi.compute() # Actually calculate NDVI
ndvi_values

<xarray.DataArray (band: 1, y: 3499, x: 7443)> Size: 104MB
array([[[-0., -0., -0., ..., -0., -0., -0.],
        [-0., -0., -0., ..., -0., -0., -0.],
        [-0., -0., -0., ..., -0., -0., -0.],
        ...,
        [-0., -0., -0., ..., -0., -0., -0.],
        [-0., -0., -0., ..., -0., -0., -0.],
        [-0., -0., -0., ..., -0., -0., -0.]]],
      shape=(1, 3499, 7443), dtype=float32)
Coordinates:
  * band         (band) int64 8B 1
  * x            (x) float64 60kB 6.22e+05 6.22e+05 ... 6.224e+05 6.224e+05
  * y            (y) float64 28kB 7.525e+06 7.525e+06 ... 7.525e+06 7.525e+06
    spatial_ref  int64 8B 0

In [ ]:
nir.data # Returns the dask array of chunks (lazy eval)
nir.values # Does the computation and actually gets the NIR values

array([[[-10000., -10000., -10000., ..., -10000., -10000., -10000.],
        [-10000., -10000., -10000., ..., -10000., -10000., -10000.],
        [-10000., -10000., -10000., ..., -10000., -10000., -10000.],
        ...,
        [-10000., -10000., -10000., ..., -10000., -10000., -10000.],
        [-10000., -10000., -10000., ..., -10000., -10000., -10000.],
        [-10000., -10000., -10000., ..., -10000., -10000., -10000.]]],
      shape=(1, 3499, 7443), dtype=float32)

In [ ]:
# The order of operations can influnce how dask distributes and schedules tasks
# Need to choose an appropriate chunk size:
#   - As big as possible without being able to read in
#   - Too small and your set-up costs might exceed your compute costs

# Once you start having to move data between nodes "shuffle" you need to be careful about whether your nodes are sharing memory or if data transfer will be expensive